# Visdom in a notebook

<!-- Copyright 2017-present, The Visdom Authors. All rights reserved.
     This source code is licensed under the license found in the
     LICENSE file in the root directory of this source tree. -->

Two ways to get a dashboard into an output cell:

- `%visdom`, the line magic, which starts a server on this machine if
  nothing is answering on the port;
- `vis.show()`, on a client you already have.

Run the cells in order. Nothing here needs a browser tab open.

## Install

`pip install visdom[notebook]` installs visdom and IPython together.
visdom itself never imports IPython -- the magic is registered against
the shell that `%load_ext` hands it -- so an existing Jupyter install
needs nothing but `pip install visdom`.

## The magic

`%visdom` embeds the dashboard for one environment. If no server
answers on the port, it starts one as a child of this kernel and waits
for it, printing where that server's log went. The environment is
registered on the way, so the frame shows the dashboard instead of the
`404` an unknown environment answers with.

In [ ]:
%load_ext visdom.notebook

In [ ]:
%visdom --env notebook-demo --height 420

## Plot into it

The frame above is the live dashboard, so this cell's plots appear in
it as they are sent. The client is an ordinary `Visdom`, pointed at the
same server and environment.

In [ ]:
import time

import numpy as np

from visdom import Visdom

vis = Visdom(env="notebook-demo")

loss = 2.0
for step in range(40):
    loss = loss * 0.93 + np.random.rand() * 0.05
    vis.line(
        X=np.array([step]),
        Y=np.array([loss]),
        win="loss",
        update="append" if step else None,
        opts=dict(title="training loss", xlabel="step"),
    )
    time.sleep(0.05)

## `vis.show()`

A client that already exists can embed itself, at whatever size suits
the cell. A bare `vis` in a cell does the same thing.

In [ ]:
vis.show(height=320)

In [ ]:
vis

## Notes

- A server on another host is embedded if it is up and never started
  from here: `%visdom --server http://gpu-box --port 8097`.
- `--no-start` embeds whatever is already running and nothing else.
- Behind a proxy, pass the prefix the server was started with:
  `%visdom --base_url /proxy/8097`.
- A server started with `-enable_login` serves its login form inside
  the frame.
- JupyterLab over HTTPS blocks a plain-HTTP frame to another port;
  `jupyter-server-proxy` in front of the server is the way through.
- In Colab the kernel's port is reached through Colab's own proxy,
  which `vis.show()` and the magic both ask for automatically.
- A server the magic started is asked to shut down, and so saves its
  environments, when this kernel exits.